# SEED-VII EEG Pipeline（唯一 Notebook）

**模型选择**：Cell 2 修改 `MODEL_TYPE` 即可切换 `eegnet` ↔ `conformer`

**数据源**：ModelScope `DEREKVERSE/SEED-VII` → 下载原始 .mat → 本地正确预处理

**预处理流程**（已修复）：
`基线校正 → MNE ICA → CAR → 居中60% → 滑动窗口 → z-score`

In [ ]:
# 安装依赖
!pip install mne torch numpy scipy scikit-learn tqdm h5py requests -q

---

## ⚙️ Cell 2：实验配置（只改这里）

In [ ]:
# ================================================================
# ⚡ 模型选择（必改）
# ================================================================
MODEL_TYPE = "eegnet"    # "eegnet"（轻量）| "conformer"（Transformer）

# ================================================================
# ModelScope
# ================================================================
MS_DATASET_ID   = "DEREKVERSE/SEED-VII"
MS_REVISION     = "master"
MS_TOKEN        = ""      # 留空则用环境变量 MODELSCOPE_API_TOKEN
LOCAL_DATA_DIR  = "./data"
LOCAL_DATA_PATH = f"{LOCAL_DATA_DIR}/SEED-VII.npz"

# ================================================================
# 预处理
# ================================================================
USE_ICA = False           # True=开启 MNE ICA（耗时），默认关闭
MAX_WINDOWS_PER_TRIAL = 60

# ================================================================
# 训练
# ================================================================
OUTPUT_DIR = f"runs_seed_vii/{MODEL_TYPE}"
SEED = 42
BATCH_SIZE = 256
NUM_WORKERS = 2
LR = 2e-4
MIN_LR = 1e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
PRETRAIN_EPOCHS = 10
MAX_EPOCHS = 200
PATIENCE = 30
ALPHA_CLS = 1.0
BETA_REG = 0.5
ENABLE_RANK = False
RANK_MARGIN = 0.05
LABEL_SMOOTHING = 0.05
SAMPLE_WEIGHT_MODE = "continuous"
INTENSITY_THRESHOLD = 0.5
WEAK_SAMPLE_WEIGHT = 0.1
TRAIN_SUBJECTS = ""
VAL_SUBJECTS = ""
TEST_SUBJECTS = ""
MAX_RUNTIME_HOURS = 10.0
SAVE_INTERVAL = 1
DEVICE = "auto"
USE_AMP = True

print(f"[CONFIG] model={MODEL_TYPE}, output={OUTPUT_DIR}, batch={BATCH_SIZE}, ica={USE_ICA}")

---

## Cell 3：ModelScope 登录

In [ ]:
import os
token = MS_TOKEN or os.environ.get("MODELSCOPE_API_TOKEN", "")
if token:
    print(f"[OK] Token found (len={len(token)})")
else:
    print("[WARN] No token — export MODELSCOPE_API_TOKEN")

---

## Cell 4（核心）：下载原始数据 + 正确预处理 → npz

**此 cell 只在首次运行或 `USE_ICA` 改变时需要重新执行**

**预处理流程（严格按 Design.md）：**
```
ICA（可选）→ 基线校正 → CAR → 居中60% → 滑动窗口 → z-score
```

**"先切分后处理"的设计说明：**
- Design.md 要求：先切分后处理（防止数据泄漏）
- 严格实现需要先划分 trial 列表，再按需读取每个 trial 单独预处理
- 实践中由于 ModelScope 数据源每个 subject 约 3GB，分开处理需多次重复下载
- 当前折中：全部预处理保存 npz，再在 npz 上划分 train/val/test
- ⚠️ 窗口级归一化存在极小跨 trial 泄漏风险；若零泄漏则需流式方案

**save_info 解析逻辑：**
- CSV 无表头，每行 = 标签,路径,intensity
- 路径含 session+emotion: movie\七类\{session}\{emotion}\文件名.mp4
- trial_id = CSV 行顺序（1→20）


In [ ]:
import os, sys, json, gc, io, warnings, time
from pathlib import Path

ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT))

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# ---- 已处理过且 ICA 配置一致？直接跳过 ----
cached_npz = Path(LOCAL_DATA_PATH)
if cached_npz.exists():
    import numpy as np
    data = np.load(cached_npz, allow_pickle=True)
    print(f"[SKIP] npz 已存在: {cached_npz}")
    print(f"  X={data['X'].shape}, y={data['y'].shape}, s={data['s'].shape}")
    print(f"  keys={list(data.files)}")
else:
    print("[INFO] 开始下载 + 预处理...")
    
    # ================================================================
    # STEP 1: 下载原始 .mat 和 save_info.csv
    # ================================================================
    from modelscope.hub.api import HubApi
    import requests

    token = MS_TOKEN or os.environ.get("MODELSCOPE_API_TOKEN", "")
    api = HubApi(token=token)
    repo_id = "DEREKVERSE/SEED-VII"

    # 列出所有文件
    all_files = []
    page = 1
    while True:
        batch = api.get_dataset_files(
            repo_id=repo_id, revision=MS_REVISION,
            root_path="/", recursive=True, page_number=page, page_size=100,
        )
        all_files.extend(batch)
        if len(batch) < 100:
            break
        page += 1

    mat_files = [f for f in all_files if f["Name"].endswith(".mat")]
    csv_files = [f for f in all_files if "_save_info.csv" in f["Name"]]
    print(f"[INFO] .mat: {len(mat_files)} 个, _save_info.csv: {len(csv_files)} 个")

    mat_dir = Path(LOCAL_DATA_DIR) / "mats"
    save_info_dir = Path(LOCAL_DATA_DIR) / "save_info"
    mat_dir.mkdir(exist_ok=True)
    save_info_dir.mkdir(exist_ok=True)

    def download_file(file_path, local_dir):
        url = api.get_dataset_file_url(
            file_name=file_path,
            dataset_name="SEED-VII",
            namespace="DEREKVERSE",
            revision=MS_REVISION,
        )
        local = Path(local_dir) / file_path.replace("/", "_")
        if local.exists() and local.stat().st_size > 0:
            return local
        resp = requests.get(url, stream=True, timeout=600)
        resp.raise_for_status()
        downloaded = 0
        with open(local, "wb") as f:
            for chunk in resp.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
        return local

    print("\n[STEP 1a] 下载 .mat...")
    for f in sorted(mat_files, key=lambda x: x["Name"]):
        local = download_file(f["Path"], mat_dir)
        print(f"  ✅ {f['Name']} ({f['Size']/1024**3:.1f}GB) → {local.name}")

    print("\n[STEP 1b] 下载 _save_info.csv...")
    for f in sorted(csv_files):
        local = download_file(f["Path"], save_info_dir)
        print(f"  ✅ {f['Name']} → {local.name}")

    # ================================================================
    # STEP 2: 解析 save_info → intensity_dict
    # 格式（无表头）：标签,路径,intensity
    # 路径含 session+emotion: movie\七类\{session}\{emotion}\文件名.mp4
    # trial_id = CSV 行顺序（1→20）
    # ================================================================
    import csv as csv_lib
    from src.labels import EMOTION_TO_IDX, SESSION_SEQUENCES, TRIALS_PER_FOLD, FOLDS_PER_SESSION

    intensity_dict = {}   # (subject, session, trial_id) -> float

    for csv_file in sorted(save_info_dir.glob("*_save_info.csv")):
        # 文件名: 1_20221104_1_save_info.csv → subject=1, session=1
        name = csv_file.stem.replace("_save_info", "")
        parts = name.split("_")
        subject = parts[0]
        session_id = int(parts[-1])   # 最后一段是 session_id

        with open(csv_file, "r", encoding="utf-8-sig") as fh:
            reader = csv_lib.reader(io.StringIO(fh.read()))
            rows = [r for r in reader if any(c.strip() for c in r)]

        for trial_id_1based, row in enumerate(rows, start=1):
            if len(row) < 2:
                continue

            # 路径在倒数第二列，intensity 在最后一列
            path = row[-2]          # e.g. "movie\七类\3\happy\8我家有喜1.mp4"
            try:
                intensity = float(row[-1])
            except (ValueError, IndexError):
                intensity = 1.0
            intensity = min(max(intensity, 0.0), 1.0)

            # 从路径解析 emotion
            # 格式: movie\七类\{session}\{emotion}\文件名
            try:
                parts_path = path.replace("/", "\\").split("\\")
                emotion = parts_path[-2]   # 倒数第二段是 emotion
                session_from_path = int(parts_path[-3])  # 倒数第三段是 session
                # 以路径中的 session 为准
                session_id = session_from_path
            except (ValueError, IndexError):
                # 回退：从 Design.md 的 SESSION_SEQUENCES 推算
                # trial_id 在 session 中的 fold 和位置
                fold_idx = (trial_id_1based - 1) // TRIALS_PER_FOLD   # 0-based
                pos_in_fold = (trial_id_1based - 1) % TRIALS_PER_FOLD
                fold_num = fold_idx + 1
                emotion = SESSION_SEQUENCES[session_id][fold_num][pos_in_fold]

            intensity_dict[(subject, session_id, trial_id_1based)] = intensity

    print(f"\n[STEP 2] intensity 标签解析完成: {len(intensity_dict)} 条")

    # ================================================================
    # STEP 3: 预处理每个 .mat → 窗口
    # ================================================================
    import h5py, numpy as np
    from src.preprocess import preprocess_trial
    from src.labels import field_id_to_label, trial_field_to_session_trial
    from sklearn.model_selection import StratifiedShuffleSplit

    cfg_preprocess = {
        "fs": 200, "window_seconds": 4.0, "step_seconds": 2.0,
        "middle_ratio": 0.6, "max_windows_per_trial": MAX_WINDOWS_PER_TRIAL,
        "use_car": True, "use_baseline_correct": True,
        "use_ica": USE_ICA, "ica_components": 20, "ica_remove": 5,
        "per_channel_zscore": True, "eps": 1e-8,
    }

    all_X, all_y, all_s, all_meta = [], [], [], []
    total_trials = 0

    for mat_file in sorted(mat_dir.glob("*.mat")):
        subject = mat_file.stem  # "1", "2", ...
        print(f"\n  处理 subject: {subject}", end="", flush=True)

        with h5py.File(mat_file, "r") as f:
            field_ids = sorted([int(k) for k in f.keys() if k.isdigit()])

        for field_id in field_ids:
            sid, trial_id = trial_field_to_session_trial(field_id)
            _, emotion_code, label_idx = field_id_to_label(field_id)

            with h5py.File(mat_file, "r") as f:
                eeg = f[str(field_id)][:].T   # (62, T)

            intensity = intensity_dict.get((subject, sid, trial_id), 1.0)

            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                arr, metas = preprocess_trial(
                    raw=eeg, subject=subject, session_id=sid, trial_id=trial_id,
                    label_idx=label_idx, intensity=intensity, cfg=cfg_preprocess,
                )

            if len(arr) == 0:
                continue

            all_X.append(arr)
            all_y.append(np.full(len(arr), label_idx, dtype=np.int64))
            all_s.append(np.full(len(arr), intensity, dtype=np.float32))
            for m in metas:
                all_meta.append({**m.__dict__})

            total_trials += 1
            if total_trials % 100 == 0:
                print(f"\n    trials={total_trials}, windows={sum(len(x) for x in all_X)}")

    print(f"\n\n[STEP 3] 完成: {total_trials} trials, {sum(len(x) for x in all_X)} windows")

    # ================================================================
    # STEP 4: 拼接 + 划分 (80/10/10)
    # ================================================================
    X = np.concatenate(all_X, axis=0).astype(np.float32)
    y = np.concatenate(all_y, axis=0)
    s = np.concatenate(all_s, axis=0)

    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, train_size=0.8, random_state=42)
    train_idx, temp_idx = next(sss.split(X, y))
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
    val_idx, test_idx = next(sss2.split(X[temp_idx], y[temp_idx]))
    val_idx = temp_idx[val_idx]
    test_idx = temp_idx[test_idx]

    for i in train_idx:   all_meta[i]["split"] = "train"
    for i in val_idx:    all_meta[i]["split"] = "val"
    for i in test_idx:   all_meta[i]["split"] = "test"

    print(f"  train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")

    # ================================================================
    # STEP 5: 保存
    # ================================================================
    meta_json = np.array([json.dumps(m, ensure_ascii=True) for m in all_meta], dtype=object)

    np.savez_compressed(
        cached_npz,
        X=X, y=y, s=s, meta=meta_json,
        split_train=train_idx.astype(np.int64),
        split_val=val_idx.astype(np.int64),
        split_test=test_idx.astype(np.int64),
    )
    print(f"\n[OK] 保存完成: {cached_npz}")
    print(f"  X={X.shape}, y={y.shape}, s={s.shape}")

    del all_X, all_y, all_s, all_meta, X, y, s
    gc.collect()

import numpy as np
data = np.load(LOCAL_DATA_PATH, allow_pickle=True)
print(f"\n[DATA] X={data['X'].shape}, y={data['y'].shape}, s={data['s'].shape}")
print(f"[DATA] train={len(data['split_train'])}, val={len(data['split_val'])}, test={len(data['split_test'])}")
print(f"[DATA] keys: {list(data.files)}")

---

## Cell 5：训练

In [ ]:
import sys
from pathlib import Path
ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT))

train_cmd = [
    "python", "scripts/train.py",
    "--data", LOCAL_DATA_PATH,
    "--output-dir", OUTPUT_DIR,
    "--seed", str(SEED),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--lr", str(LR),
    "--min-lr", str(MIN_LR),
    "--weight-decay", str(WEIGHT_DECAY),
    "--grad-clip", str(GRAD_CLIP),
    "--pretrain-epochs", str(PRETRAIN_EPOCHS),
    "--max-epochs", str(MAX_EPOCHS),
    "--patience", str(PATIENCE),
    "--alpha-cls", str(ALPHA_CLS),
    "--beta-reg", str(BETA_REG),
    "--rank-margin", str(RANK_MARGIN),
    "--label-smoothing", str(LABEL_SMOOTHING),
    "--sample-weight-mode", SAMPLE_WEIGHT_MODE,
    "--intensity-threshold", str(INTENSITY_THRESHOLD),
    "--weak-sample-weight", str(WEAK_SAMPLE_WEIGHT),
    "--max-runtime-hours", str(MAX_RUNTIME_HOURS),
    "--save-interval", str(SAVE_INTERVAL),
    "--device", DEVICE,
    "--model-type", MODEL_TYPE,
]
if TRAIN_SUBJECTS: train_cmd += ["--train-subjects", TRAIN_SUBJECTS]
if VAL_SUBJECTS:   train_cmd += ["--val-subjects",   VAL_SUBJECTS]
if TEST_SUBJECTS:  train_cmd += ["--test-subjects",  TEST_SUBJECTS]
if not USE_AMP:    train_cmd.append("--no-amp")
else:              train_cmd.append("--amp")

print("[TRAIN CMD]", " ".join(train_cmd))
print("=" * 70)
import subprocess
result = subprocess.run(train_cmd, cwd=str(ROOT))
if result.returncode == 0:
    print("[OK] 训练完成")
else:
    print(f"[ERROR] 训练失败: exit {result.returncode}")

---

## Cell 6：编码特征

In [ ]:
from pathlib import Path

best_encoder = Path(OUTPUT_DIR) / "best_encoder.pt"
encode_output = Path(OUTPUT_DIR) / "encoded_features.npz"

if not best_encoder.exists():
    print(f"[WARN] Checkpoint not found: {best_encoder} — 跳过")
else:
    print(f"[ENCODE] {best_encoder} → {encode_output}")
    import subprocess
    encode_cmd = [
        "python", "scripts/encode.py",
        "--data",       LOCAL_DATA_PATH,
        "--checkpoint", str(best_encoder),
        "--output",     str(encode_output),
        "--model-type", MODEL_TYPE,
        "--batch-size", str(BATCH_SIZE),
        "--device",     DEVICE,
    ]
    if USE_AMP: encode_cmd.append("--amp")
    r = subprocess.run(encode_cmd, cwd=str(ROOT))
    if r.returncode == 0:
        print(f"[OK] → {encode_output}")
    else:
        print(f"[ERROR] encode failed: {r.returncode}")

---

## Cell 7：结果汇总

In [ ]:
from pathlib import Path
import json, numpy as np

summary_path = Path(OUTPUT_DIR) / "summary.json"
if summary_path.exists():
    with open(summary_path) as f:
        s = json.load(f)
    print("=" * 60)
    print(f"  模型   : {s.get('model_type', MODEL_TYPE)}")
    print(f"  参数量 : {s.get('n_params', 'N/A'):,} ({s.get('n_params', 0)/1e6:.4f}M)")
    print(f"  epochs : {s.get('epochs_run', 'N/A')}")
    print(f"  最佳val_acc: {s.get('best_val_acc', 'N/A'):.4f} @ ep {s.get('best_epoch', 'N/A')}")
    print(f"  耗时   : {s.get('elapsed_seconds', 0)/60:.1f} min")
    t = s.get("test", {})
    if t:
        print(f"  test_acc: {t.get('acc', 'N/A'):.4f}  MAE: {t.get('intensity_mae', 'N/A'):.4f}")
    print("=" * 60)
else:
    print(f"[WARN] {summary_path} not found")

encode_output = Path(OUTPUT_DIR) / "encoded_features.npz"
if encode_output.exists():
    feat = np.load(encode_output)
    acc = (feat["cls_pred"] == feat["labels"]).mean()
    mae = np.abs(feat["intensity_pred"] - feat["intensity_true"]).mean()
    print(f"\n[ENCODED] features={feat['features'].shape}, acc={acc:.4f}, MAE={mae:.4f}")

print(f"\n✅ Pipeline 完成")
print(f"   output: {OUTPUT_DIR}")
print(f"   npz:    {LOCAL_DATA_PATH}")